# Day 45 — End-to-end modeling mini-project
Goal: deliver a reproducible pipeline from data → model → evaluation → saved artifacts.

## Checklist
- Load dataset and split train/valid/test.
- Preprocess with ColumnTransformer.
- Train baseline model.
- Evaluate with appropriate metrics; cross-validate.
- Save model + preprocessing (joblib).
- Write a short README section in this notebook.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
artifact_dir = Path('artifacts/day45')
artifact_dir.mkdir(parents=True, exist_ok=True)
df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','fare','age'])
X = df[['sex','class','fare','age']]
y = df['survived']
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                           ('num', StandardScaler(), ['fare','age'])])
pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
scores = cross_val_score(pipe, Xtr, ytr, cv=5, scoring='roc_auc')
scores.mean()
pipe.fit(Xtr,ytr); pipe.score(Xte,yte)
joblib.dump(pipe, artifact_dir / 'titanic_pipeline.joblib')


## Deliverables
- artifacts/day45/titanic_pipeline.joblib
- This notebook with metrics, rationale, and next steps.
- (Optional) a minimal FastAPI app for prediction.

## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — an end-to-end evidence chain from decision to reproducible artifact

### Mental model

An end-to-end project is a sequence of contracts, not one long notebook:
problem framing defines the decision and prediction time; data
validation defines row grain and allowed values; splitting protects the
evaluation boundary; a pipeline binds preprocessing to the model;
metrics and error analysis support a scoped claim; artifact packaging
preserves the exact fitted workflow.

A baseline is a decision checkpoint. Added complexity is justified only
if it improves a declared metric or operational property under the same
data and evaluation design. Reproducibility also requires data identity,
environment, seed, commands, and limitations.

### Read the API before running it

- **`DummyClassifier(strategy=...)`:** creates a minimal predictive reference under the exact same split and metric.
- **`ColumnTransformer` inside `Pipeline`:** binds column-specific preprocessing and prediction into one fitted artifact.
- **manifest + acceptance gates:** connect data hash, schema, code/environment, metrics, limitations, and artifact identity.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — require the candidate to beat a same-split baseline

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** ROC AUC and this split reflect the decision; the test result was not repeatedly consulted during development.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=4501
)
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
candidate = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2_000)
).fit(X_train, y_train)
scores = {
    "baseline": roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1]),
    "candidate": roc_auc_score(y_test, candidate.predict_proba(X_test)[:, 1]),
}
print(scores)
assert scores["candidate"] > scores["baseline"]

**Expected observation:** The model must improve the declared held-out metric over the simple prior baseline on identical rows.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — turn completion into explicit acceptance gates

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Thresholds and evidence requirements were set before viewing the final candidate result.

In [ ]:
evidence = {
    "data_contract_passed": True,
    "tests_passed": True,
    "baseline_auc": 0.50,
    "candidate_auc": 0.97,
    "minimum_improvement": 0.05,
    "artifact_reloaded": True,
    "limitations_documented": True,
}
gates = {
    "quality": evidence["data_contract_passed"] and evidence["tests_passed"],
    "performance": (
        evidence["candidate_auc"] - evidence["baseline_auc"]
        >= evidence["minimum_improvement"]
    ),
    "delivery": evidence["artifact_reloaded"] and evidence["limitations_documented"],
}
print(gates)
assert all(gates.values())

**Expected observation:** A project is ready only when every named quality, performance, and delivery gate passes.

### Debugging and practice ramp

**Common mistake:** Treating a high notebook score as the project outcome while omitting baseline, leakage audit, artifact reload, and limitations.

**Diagnostic:** Trace every claim backward to metric rows, split, feature pipeline, data version, environment, and command; rerun in a fresh kernel/process.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define an end-to-end evidence chain from decision to reproducible artifact in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not promote a project when a gate is unknown, final evaluation influenced iteration, or the intended decision and harm boundaries are vague.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

The notebook's project checklist is the exercise:

1. Load the dataset and create train/validation/test boundaries.

**Verify:** Practice 1 — an end-to-end evidence chain from decision to reproducible artifact — record dataset identity/hash, target, row count, and frozen train/validation/test indices; assert the three index sets are pairwise disjoint and their counts sum to the validated rows.

2. Preprocess with `ColumnTransformer`.

**Verify:** Practice 2 — an end-to-end evidence chain from decision to reproducible artifact — fit one ColumnTransformer inside the training pipeline, print output feature names/order and transformed shapes, and predict a validation row containing the declared missing/unknown-category boundary.

3. Train a baseline model.

**Verify:** Practice 3 — an end-to-end evidence chain from decision to reproducible artifact — fit a declared naive/simple baseline on training rows, print validation metric and denominator/support, and save its parameters and seed before trying a more complex candidate.

4. Evaluate with appropriate metrics and cross-validation.

**Verify:** Practice 4 — an end-to-end evidence chain from decision to reproducible artifact — print every cross-validation score plus mean/std on training data and one frozen validation comparison; reserve the test set for one final evaluation and report the metric formula and support.

5. Save the model and preprocessing together with `joblib`.

**Verify:** Practice 5 — an end-to-end evidence chain from decision to reproducible artifact — save one joblib pipeline containing preprocessing and model, compute its SHA-256, reload it in a fresh process, and assert predictions and feature metadata match the pre-save values.

6. Write a short README-style section in the notebook covering rationale,
   metrics, limitations, and next steps.

**Verify:** Practice 6 — an end-to-end evidence chain from decision to reproducible artifact — include runnable setup/train/test commands, data provenance/hash, baseline/candidate metrics, limitations, and next step; have a clean-shell replay finish with exit code 0.

7. Optionally adapt the Day 44 FastAPI service.

**Verify:** Practice 7 — an end-to-end evidence chain from decision to reproducible artifact — if the API extension is attempted, run Day 44 health/valid/invalid TestClient cases against the reloaded artifact; otherwise record an explicit skipped result and keep the capstone complete.

### Progressive hints

1. Start by writing the target, row unit, and split before feature engineering.
2. Use `handle_unknown="ignore"` for categorical inference and keep every fitted
   transformation inside the pipeline.
3. Report the distribution of fold scores; do not tune against the holdout.
4. Reload the saved artifact in a fresh cell and predict a small valid batch.

### Additional mastery practice

Finish one reproducible modeling system rather than a notebook demo. Every data boundary, fitted transform, metric, artifact, and handoff claim needs evidence.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

8. **Data-contract gate:** Write executable checks for row identity, required columns, target domain, missingness limits, duplicate policy, and data snapshot fingerprint.
   **Progressive hint:** Validate raw data before splitting. Separate hard failures from reported warnings and hash stable source bytes or a canonical snapshot manifest.

**Verify:** Data-contract gate — run the contract against one valid fixture and separate duplicate-ID, missing-column, invalid-target, excessive-missingness, and changed-snapshot fixtures; assert each failure names its rule and print the accepted snapshot hash.

9. **Leakage audit:** Create a feature-by-feature table with availability time, source, transformation fit scope, and leakage decision. Investigate at least one suspicious post-outcome field.
   **Progressive hint:** Ask whether the value exists at prediction time and whether it was computed using future rows or target information.

**Verify:** Leakage audit — save a feature lineage table with feature, source, availability timestamp, fit scope, target dependence, and decision; remove/quarantine the post-outcome fixture and show the corrected split/metric.

10. **Baseline ladder:** Evaluate a dummy strategy, a simple linear/tree model, and one selected candidate on identical folds. Define a minimum practical improvement before seeing results.
   **Progressive hint:** Use paired fold scores and include runtime/complexity. A statistically detectable gain may still be operationally irrelevant.

**Verify:** Baseline ladder — print fold-level and mean/std metrics for DummyClassifier, the declared simple model, and one candidate on identical folds; record a predeclared practical-improvement threshold and whether it is met.

11. **Operating-policy selection:** Build a threshold table with false-positive cost, false-negative cost, precision, recall, and queue volume. Select a threshold on validation data, then freeze it.
   **Progressive hint:** Translate confusion-matrix counts into the same business unit and include capacity constraints such as maximum daily reviews.

**Verify:** Operating-policy selection — print one row per threshold with TP/FP/FN/TN, precision, recall, queue volume, and total expected cost; select on validation only, serialize the frozen threshold, and apply it once to test scores.

12. **Error-slice analysis:** Define at least three pre-motivated slices, report support and error metrics, and inspect representative false positives and false negatives without exposing sensitive raw values.
   **Progressive hint:** Choose slices from domain risk, not by mining the test set for the worst-looking subgroup. Small support requires uncertainty and caution.

**Verify:** Error-slice analysis — for at least three predeclared slices, print support, metric, and uncertainty plus sanitized false-positive/false-negative examples; flag slices below minimum support rather than ranking them.

13. **Artifact manifest:** Save the fitted pipeline with a JSON manifest containing model ID, training-data fingerprint, schema, metric definitions/results, threshold, dependency versions, and file hashes.
   **Progressive hint:** JSON holds metadata; joblib holds the trusted fitted object. Write both to a versioned artifacts directory and verify them on load.

**Verify:** Artifact manifest — validate the JSON manifest schema and every listed SHA-256/size, then tamper one artifact and assert loading/promotion stops before prediction.

14. **Fresh-process acceptance:** Create a smoke test that starts from a clean process, loads the saved artifact, scores a fixed fixture, and compares the result with the pre-save prediction within a numeric tolerance.
   **Progressive hint:** Do not rely on notebook variables. The test needs only documented files, installed dependencies, and repository-relative paths.

**Verify:** Fresh-process acceptance — run a subprocess from a clean working directory that loads the artifact and scores the fixed fixture; require exit code 0 and prediction parity within the declared numeric tolerance.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 8 — Data-contract gate


# Practice 9 — Leakage audit


# Practice 10 — Baseline ladder


# Practice 11 — Operating-policy selection


# Practice 12 — Error-slice analysis


# Practice 13 — Artifact manifest


# Practice 14 — Fresh-process acceptance
